# Frequency, severity and pure premium models

This notebook builds the M3 models on `mart_person_year` and explains each step for a reviewer. Several
candidate model structures predict the same quantity, the expected annual PMI-scope cost of a person-year at
2023 prices. They are compared on one yardstick before one is chosen. The notebook uses aggregated tables
only and shows no person-level rows.

**Source.** AHRQ, Medical Expenditure Panel Survey (MEPS), Household Component, full-year consolidated files
HC-243 (2022) and HC-251 (2023), with the inpatient, outpatient and office-based event files.

## Data and metrics

**Which rows.** Models are fitted on the 2022 `train` rows, and scored on the 2022 `test` rows. The test rows
are a household-grouped sample of 2022, so no household is on both sides. The 2023 rows are held back for
the out-of-time test in M5 and are not loaded here.

**What is predicted.** One row is one person-year. A prediction is the expected annual cost of that
person-year, with its exposure included, so a part-year person has a smaller expected cost. Costs are at 2023
prices. Models are fitted unweighted, because an insurer would fit on its own book.

**Factors.** The 18 rating factor candidates are typed as categories. The reference level of each is the level
with the most train exposure, and it can never be `not_applicable` or `unknown`. Age is shown in single years
here. Banding comes in the level grouping task.

**Metrics.** Every candidate is scored by the same package code.

- **Tweedie deviance** on annual cost, with power 1.5. The ranking is re-checked at 1.3 and 1.7.
- **Gini**, weighted by exposure, for how well the predictions rank the person-years.
- **Lift and double-lift** tables, which compare actual with expected cost by group.
- **A/E** (actual divided by expected) overall, by factor level and by predicted-cost decile.

In [1]:
import pandas as pd

from health_pricing.models import frame, metrics

pd.options.display.float_format = "{:,.3f}".format
frames = frame.load_frames()
frame.frame_summary(frames)

,split,person_years,exposure,claimants,claimant_rate,claims,cost,mean_cost_per_claimant
0,train,17596,"17,440.750",7229,0.411,53322,"38,080,040.270","5,267.677"
1,test,4151,"4,100.333",1731,0.417,12463,"9,748,983.247","5,631.995"


About 41 to 42 percent of person-years have at least one PMI-scope claim in both splits. Mean cost per claimant is
higher in test than in train, which is the first sign of how much a skewed cost varies between samples.

### Reference levels

Age has the most exposure at 85 only because single years are not yet banded, so its reference level is
temporary. The reference levels of the other factors are the largest groups, as intended.

In [2]:
frame.reference_table(frames)

,factor,reference_level,exposure_share
0,age,85,0.022
1,region,3,0.384
2,inscov,1,0.592
3,rthlth,2,0.325
4,mnhlth,2,0.311
5,OFTSMK53,3,0.702
6,PHYEXE53,1,0.400
7,HIBPDX,0,0.500
8,CHDDX,0,0.756
9,ANGIDX,0,0.787


### Baseline

The baseline predicts the same cost per exposure year for everyone, the train mean. Every candidate has to beat
it. Its Gini is 0 by construction, because it does not rank anyone, and its A/E on train is 1.

In [3]:
metrics.baseline_scores(frames.train, frames.test)

,split,tweedie_deviance,mean_tweedie_deviance,gini,a_e
0,train,"3,694,004.075",209.934,0.000,1.000
1,test,"928,363.961",223.648,0.000,1.089


In [4]:
baseline = metrics.baseline_predictions(frames.train, frames.test)
pd.DataFrame(metrics.deviance_by_power(frames.test["cost"], baseline), index=["Test deviance"])

,1.300,1.500,1.700
Test deviance,"4,118,785.057","928,363.961","252,758.818"


The baseline's test A/E is above 1, so test cost per exposure year is higher than train. Part of this is
sampling variation in a skewed cost. Every candidate's test A/E should be read against this starting gap.
Deviances at different powers are on different scales, so they are only compared between candidates at the
same power. The power check asks whether the ranking of candidates changes.